# 11 — Building a tiny autodiff engine

> **Status: in progress.** The outline and starter code below are complete;
> the full derivations are being written. Following along? Open an issue or a PR.

Every gradient so far was derived by hand. That does not scale — real frameworks
build a computation graph and differentiate it automatically. This chapter builds
a working reverse-mode autodiff engine in about 100 lines, then trains a network
with it and never writes a derivative again.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(42)

## What this chapter covers

1. Computation graphs: nodes are values, edges are operations
2. Forward mode vs reverse mode, and why reverse mode wins for deep learning
3. A Value class holding data, gradient, and a local backward closure
4. Operator overloading so `a * b + c` builds the graph implicitly
5. Topological sort, then propagate gradients backwards through it
6. Wrap it in Neuron / Layer / MLP classes
7. Train on a real problem and verify against chapter 04's hand-derived gradients

## Starter code

Enough to run and build on. The rest of the chapter fills in from here.

In [2]:
class Value:
    """A scalar that remembers how it was computed.

    Each Value stores its data, an accumulated gradient, and a closure that
    knows how to push gradient to its inputs. Chain them together and you
    have a computation graph you can differentiate.
    """

    def __init__(self, data, children=(), op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)
        self._op = op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            # Addition passes gradient through unchanged to both inputs.
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            # Product rule.
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def backward(self):
        # Topological order, so every node is processed after its consumers.
        topo, visited = [], set()

        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)

        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
d = a * b + c          # d = -6 + 10 = 4
d.backward()

print("d =", d)
print(f"dd/da = {a.grad}  (expected {b.data})")
print(f"dd/db = {b.grad}  (expected {a.data})")
print(f"dd/dc = {c.grad}  (expected 1.0)")
print("\nTODO: add tanh, exp, pow, then build Neuron / Layer / MLP on top.")

d = Value(data=4.0000, grad=1.0000)
dd/da = -3.0  (expected -3.0)
dd/db = 2.0  (expected 2.0)
dd/dc = 1.0  (expected 1.0)

TODO: add tanh, exp, pow, then build Neuron / Layer / MLP on top.


## Exercises

Each section above becomes an exercise until the chapter is written out in full.
Implement one, check it against `numerical_gradient` or a `scikit-learn`
reference where applicable, and open a PR if you want it reviewed.

---

**You've reached the end of the current chapters.** New chapters are added as I work through them — watch the repo or open an issue with a request.